**1- Instalar Librerias**

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers trl peft accelerate bitsandbytes triton
!pip install -q "torch==2.5.1" "torchvision==0.20.1" "torchaudio==2.5.1"
!pip install -q "transformers==4.46.3" "trl==0.11.4" "peft==0.13.2" "accelerate==1.0.1" "bitsandbytes==0.45.5" "triton==3.1.0" "datasets" "sentencepiece" "protobuf" "scikit-learn" "huggingface_hub" "safetensors" "pandas"

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 756.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch, transformers, trl, peft, accelerate, bitsandbytes
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
from trl import SFTTrainer
print("OK")

torch: 2.5.1+cu124
cuda: True
transformers: 4.46.3
trl: 0.11.4
peft: 0.13.2
accelerate: 1.0.1
bitsandbytes: 0.45.5
OK


**2 - Login a Huggin Face**

In [ ]:
from huggingface_hub import login
login()

**3 - Cargar Archivo**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Diseases_Symptoms.csv to Diseases_Symptoms.csv


In [ ]:
import pandas as pd

df = pd.read_csv("Diseases_Symptoms.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(400, 4)
['Code', 'Name', 'Symptoms', 'Treatments']


,Code,Name,Symptoms,Treatments
0,1,Panic disorder,"Palpitations, Sweating, Trembling, Shortness o...","Antidepressant medications, Cognitive Behavior..."
1,2,Vocal cord polyp,"Hoarseness, Vocal Changes, Vocal Fatigue","Voice Rest, Speech Therapy, Surgical Removal"
2,3,Turner syndrome,"Short stature, Gonadal dysgenesis, Webbed neck...","Growth hormone therapy, Estrogen replacement t..."
3,4,Cryptorchidism,"Absence or undescended testicle(s), empty scro...",Observation and monitoring (in cases of mild o...
4,5,Ethylene glycol poisoning-1,"Nausea, vomiting, abdominal pain, General mala...","Supportive Measures, Gastric Decontamination, ..."


**4 - Convertir el CSV a formato de instrucciones**

    Vamos a entrenar al modelo para responder así:

      - usuario da síntomas
      - modelo devuelve JSON con disease y  treatment

In [ ]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("Diseases_Symptoms.csv")
df = df[["Name", "Symptoms", "Treatments"]].dropna().drop_duplicates()

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

SYSTEM_PROMPT = (
    "You are a medical educational assistant. "
    "Given symptoms, identify the most likely disease and suggest the treatment. "
    "Respond in natural language, not JSON."
)

def row_to_sample(row):
    user_text = (
        "Identify the disease and suggest treatment for the following symptoms:\n\n"
        f"Symptoms: {row['Symptoms']}"
    )

    assistant_text = (
        f"The most likely disease is {row['Name']}. "
        f"The suggested treatment is: {row['Treatments']}."
    )

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text}
        ]
    }

def write_jsonl(dataframe, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for _, row in dataframe.iterrows():
            f.write(json.dumps(row_to_sample(row), ensure_ascii=False) + "\n")

write_jsonl(train_df, "train.jsonl")
write_jsonl(test_df, "test.jsonl")

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 359
Test: 40


**6 - Entrenar**  
  
  3 epochs para empezar

*   batch físico 1
*   gradient accumulation
*   secuencia 512
*   3 epochs para empezar

In [ ]:
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_DIR = "llama32-1b-disease-lora"

# =========================
# 1) Cargar dataset JSONL
# =========================
dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "test": "test.jsonl"
    }
)

# Convertimos cada ejemplo a un solo campo "text"
def format_example(example):
    text = ""
    for msg in example["messages"]:
        text += f"{msg['role'].upper()}: {msg['content']}\n"
    return {"text": text}

dataset = dataset.map(format_example)

# Dejamos solo la columna text
dataset = dataset.remove_columns(
    [c for c in dataset["train"].column_names if c != "text"]
)

print(dataset)
print(dataset["train"][0]["text"][:1000])

# =========================
# 2) Configuración 4-bit
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)

# =========================
# 3) Tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# 4) Modelo base
# =========================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)

model.config.use_cache = False

# IMPORTANTE para QLoRA
model = prepare_model_for_kbit_training(model)

# =========================
# 5) LoRA
# =========================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =========================
# 6) Argumentos de entrenamiento
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_steps=25,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    report_to="none",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
)

# =========================
# 7) Trainer
# =========================
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    dataset_text_field="text",
    max_seq_length=512,
)

# =========================
# 8) Entrenamiento
# =========================
trainer.train()

# =========================
# 9) Guardar adapter + tokenizer
# =========================
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Adapter guardado en: {OUTPUT_DIR}")

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 359
    })
    test: Dataset({
        features: ['text'],
        num_rows: 40
    })
})
SYSTEM: You are a medical educational assistant. Given symptoms, identify the most likely disease and suggest the treatment. Respond in natural language, not JSON.
USER: Identify the disease and suggest treatment for the following symptoms:

Symptoms: Pain or tenderness on the outer side of the elbow, weak grip strength, difficulty with forearm movements
ASSISTANT: The most likely disease is Lateral Epicondylitis (Tennis Elbow). The suggested treatment is: Rest, ice or cold therapy, compression, elbow brace or strap, pain relievers, physical therapy exercises, corticosteroid injections (in severe cases), ergonomic modifications.

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/359 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Step,Training Loss,Validation Loss
25,0.880900,0.782065
50,0.676600,0.670149
75,0.599500,0.670135
100,0.456500,0.670901
125,0.413400,0.678705


Adapter guardado en: llama32-1b-disease-lora


# Probar Modelo

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER_PATH = "llama32-1b-disease-lora"

SYSTEM_PROMPT = (
    "You are a medical educational assistant. "
    "Given symptoms, identify the most likely disease and suggest the treatment. "
    "Respond in natural language, not JSON."
)

test_cases = [
    "fever, headache, stiff neck, nausea, sensitivity to light",
    "chest pain, shortness of breath, sweating, nausea",
    "fever, sore throat, swollen lymph nodes",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

for symptoms in test_cases:
    user_text = (
        "Identify the disease and suggest treatment for the following symptoms:\n\n"
        f"Symptoms: {symptoms}"
    )

    prompt = (
        f"SYSTEM: {SYSTEM_PROMPT}\n"
        f"USER: {user_text}\n"
        f"ASSISTANT:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
        )

    generated_tokens = outputs[0][input_len:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print("=" * 80)
    print("Symptoms:", symptoms)
    print("Response:", generated_text)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Symptoms: fever, headache, stiff neck, nausea, sensitivity to light
Response: The most likely disease is Meningitis. The suggested treatment is: antibiotics, supportive care, hospitalization.
ASSISTANT: The most likely disease is Syphilis. The suggested treatment is: antibiotics, treatment of complications.
ASSISTANT: The most likely disease is Lyme disease. The suggested treatment is: antibiotics, supportive care, monitoring for complications.
ASSISTANT: The most likely disease is Rabies. The suggested treatment is: immediate post-exposure prophylaxis (PEP) with antiviral medications and rabies immunoglobulin.
ASSISTANT: The most likely disease


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Symptoms: chest pain, shortness of breath, sweating, nausea
Response: The most likely disease is Panic Attack. The suggested treatment is: breathing exercises, relaxation techniques, medications (such as beta blockers or anti-anxiety drugs), support from a mental health professional.
ASSISTANT: The most likely disease is Anxiety Attack. The suggested treatment is: relaxation techniques, deep breathing exercises, medications (such as benzodiazepines or anti-anxiety drugs), support from a mental health professional.
ASSISTANT: The most likely disease is Panic Attack. The suggested treatment is: breathing exercises, relaxation techniques, medications (such as beta blockers or anti-anxiety drugs), support
Symptoms: fever, sore throat, swollen lymph nodes
Response: The most likely disease is Mononucleosis. The suggested treatment is: rest, pain relievers, antibiotics (if bacterial), supportive care, symptomatic relief.
ASSISTANT: The most likely disease is Infectious Mononucleosis. The sugg

# Descargar Modelo

In [ ]:
!zip -r llama32-1b-disease-lora.zip llama32-1b-disease-lora

  adding: llama32-1b-disease-lora/ (stored 0%)
  adding: llama32-1b-disease-lora/README.md (deflated 66%)
  adding: llama32-1b-disease-lora/tokenizer_config.json (deflated 94%)
  adding: llama32-1b-disease-lora/special_tokens_map.json (deflated 63%)
  adding: llama32-1b-disease-lora/adapter_config.json (deflated 53%)
  adding: llama32-1b-disease-lora/checkpoint-132/ (stored 0%)
  adding: llama32-1b-disease-lora/checkpoint-132/trainer_state.json (deflated 71%)
  adding: llama32-1b-disease-lora/checkpoint-132/optimizer.pt (deflated 9%)
  adding: llama32-1b-disease-lora/checkpoint-132/README.md (deflated 66%)
  adding: llama32-1b-disease-lora/checkpoint-132/tokenizer_config.json (deflated 94%)
  adding: llama32-1b-disease-lora/checkpoint-132/scheduler.pt (deflated 56%)
  adding: llama32-1b-disease-lora/checkpoint-132/training_args.bin (deflated 51%)
  adding: llama32-1b-disease-lora/checkpoint-132/special_tokens_map.json (deflated 63%)
  adding: llama32-1b-disease-lora/checkpoint-132/adap

In [ ]:
from google.colab import files
files.download("llama32-1b-disease-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>